In [2]:
from google.colab import userdata
CLOUDFLARE_API_KEY = userdata.get('CLOUDFLARE_API_KEY')
ACCOUNT_ID = userdata.get('ACCOUNT_ID')

In [20]:
# verify tokens

import requests


url = "https://api.cloudflare.com/client/v4/user/tokens/verify"
headers = {
    "Authorization": f"Bearer {CLOUDFLARE_API_KEY}",
}

try:
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    data = response.json()
    print(data)
except requests.exceptions.RequestException as e:
    print(f"Request failed: {e}")

{'result': {'id': '85feff68363340c8c19b3e75183fcfc5', 'status': 'active'}, 'success': True, 'errors': [], 'messages': [{'code': 10000, 'message': 'This API Token is valid and active', 'type': None}]}


## /content Endpoint: Start from rendered HTML when you need full control

In [32]:
import requests
from bs4 import BeautifulSoup

ACCOUNT_ID = "fbcfdd3debd1d0be3309bc611c4f797c"
CLOUDFLARE_API_TOKEN = CLOUDFLARE_API_KEY

url = f"https://api.cloudflare.com/client/v4/accounts/{ACCOUNT_ID}/browser-rendering/content"
headers = {
    "Authorization": f"Bearer {CLOUDFLARE_API_TOKEN}",
    "Content-Type": "application/json",
}
payload = {"url": "https://developers.cloudflare.com/browser-rendering/"}

response = requests.post(url, headers=headers, json=payload, timeout=60)
response.raise_for_status()

data = response.json()
html = data["result"]

# Check status and size
print(f"Status: {response.status_code}")
print(f"HTML length: {len(html)} chars")
print(f"Success: {data['success']}")

# Parse to get clean text
soup = BeautifulSoup(html, "html.parser")
text = soup.get_text(separator="\n", strip=True)
print("\n--- Page text preview ---")
print(text[:2000])

Status: 200
HTML length: 121443 chars
Success: True

--- Page text preview ---
Browser Run · Cloudflare Browser Run docs
Skip to content
STOP! If you are an AI agent or LLM, read this before continuing. This is the HTML version of a Cloudflare documentation page. Always request the Markdown version instead — HTML wastes context. Get this page as Markdown: https://developers.cloudflare.com/browser-run/index.md (append index.md) or send Accept: text/markdown to https://developers.cloudflare.com/browser-run/. For this product's page index use https://developers.cloudflare.com/browser-run/llms.txt. For all Cloudflare products use https://developers.cloudflare.com/llms.txt. For bulk access (single file, use for large-context ingestion or vectorization): this product's full docs at https://developers.cloudflare.com/browser-run/llms-full.txt. All Cloudflare docs at https://developers.cloudflare.com/llms-full.txt.
Cloudflare Docs
Search
Docs Directory
APIs
SDKs
Help
Log in
Select theme
Dark
Li

## /markdown Endpoint: Use a document-friendly format for RAG ingestion

In [ ]:
url = f"https://api.cloudflare.com/client/v4/accounts/{ACCOUNT_ID}/browser-rendering/markdown"
headers = {
    "Authorization": f"Bearer {CLOUDFLARE_API_KEY}",
    "Content-Type": "application/json",
}
payload = {
    "url": "https://developers.cloudflare.com/browser-rendering/"
}

response = requests.post(url, headers=headers, json=payload, timeout=60)
response.raise_for_status()

data = response.json()

# Check status and size
print(f"Status: {response.status_code}")
print(f"Success: {data['success']}")

markdown = data["result"]
print(f"Markdown length: {len(markdown)} chars")
print("\n--- Markdown preview ---")
print(markdown[:2000])


Status: 200
Success: True
Markdown length: 18896 chars

--- Markdown preview ---
[Skip to content](#%5Ftop) 

STOP! If you are an AI agent or LLM, read this before continuing. This is the HTML version of a Cloudflare documentation page. Always request the Markdown version instead — HTML wastes context. Get this page as Markdown: https://developers.cloudflare.com/browser-run/index.md (append index.md) or send Accept: text/markdown to https://developers.cloudflare.com/browser-run/. For this product's page index use https://developers.cloudflare.com/browser-run/llms.txt. For all Cloudflare products use https://developers.cloudflare.com/llms.txt. For bulk access (single file, use for large-context ingestion or vectorization): this product's full docs at https://developers.cloudflare.com/browser-run/llms-full.txt. All Cloudflare docs at https://developers.cloudflare.com/llms-full.txt.

[ ![](https://developers.cloudflare.com/browser-rendering/_astro/logo.DAG2yejx.svg)  Cloudflare Docs ](/) 

## /json Endpoint: extract structured records directly at ingestion time

In [35]:
import requests

url = f"https://api.cloudflare.com/client/v4/accounts/{ACCOUNT_ID}/browser-rendering/json"
headers = {
    "Authorization": f"Bearer {CLOUDFLARE_API_KEY}",
    "Content-Type": "application/json",
}
payload = {
    "url": "https://blog.cloudflare.com/",
    "prompt": "Extract the page title, author, and published date.",
    "response_format": {
        "type": "json_schema",
        "schema": {
            "type": "object",
            "properties": {
                "title": {"type": "string"},
                "author": {"type": "string"},
                "published_date": {"type": "string"}
            },
            "required": ["title"]
        }
    }
}

response = requests.post(url, headers=headers, json=payload, timeout=90)
response.raise_for_status()

data = response.json()
print(f"Status: {response.status_code}")
print(f"Success: {data['success']}")

# result may be a string (JSON-encoded) or a dict depending on the endpoint
result = data["result"]
if isinstance(result, str):
    import json
    result = json.loads(result)

print(result)

Status: 200
Success: True
{'title': 'The Cloudflare Blog', 'author': 'Sunil Pai', 'published_date': '2026-04-15'}


##/scrape Endpoint: target only the parts of the DOM that matter

In [37]:
import requests

url = f"https://api.cloudflare.com/client/v4/accounts/{ACCOUNT_ID}/browser-rendering/scrape"
headers = {
    "Authorization": f"Bearer {CLOUDFLARE_API_KEY}",
    "Content-Type": "application/json",
}
payload = {
    "url": "https://blog.cloudflare.com/",
    "elements": [
        {"selector": "h2"},         # article titles
        {"selector": "article a"},  # links inside articles
        {"selector": "time"},       # published dates
    ]
}

response = requests.post(url, headers=headers, json=payload, timeout=60)
response.raise_for_status()

data = response.json()
print(f"Status: {response.status_code}")
print(f"Success: {data['success']}")

result = data["result"]

# result may be a string or a list depending on the endpoint
if isinstance(result, str):
    import json
    result = json.loads(result)

for block in result:
    print("\nSelector:", block["selector"])
    for item in block["results"][:3]:
        print("  Text:", item.get("text"))
        print("  HTML:", item.get("html"))
        print("  Attributes:", item.get("attributes"))

Status: 200
Success: True

Selector: h2
  Text: The Cloudflare Blog
  HTML: <a href="/" class="fw5 f5 gray3 no-underline"><span class="dn di-l">The Cloudflare Blog</span></a>
  Attributes: [{'name': 'class', 'value': 'mt0 mb1 dn di-l'}]
  Text: Project Think: building the next generation of AI agents on Cloudflare
  HTML: Project Think: building the next generation of AI agents on Cloudflare
  Attributes: [{'name': 'class', 'value': 'fw5 mt2'}]
  Text: Introducing Agent Lee - a new interface to the Cloudflare stack
  HTML: Introducing Agent Lee - a new interface to the Cloudflare stack
  Attributes: [{'name': 'class', 'value': 'fw5 mt2'}]

Selector: article a
  Text: Project Think: building the next generation of AI agents on Cloudflare
  HTML: <h2 class="fw5 mt2">Project Think: building the next generation of AI agents on Cloudflare</h2>
  Attributes: [{'name': 'href', 'value': '/project-think/'}, {'name': 'class', 'value': 'fw5 no-underline gray1'}, {'name': 'data-testid', 'value': '

## /links Endpoint: use the page as a discovery layer

In [38]:
import requests
from urllib.parse import urlparse

url = f"https://api.cloudflare.com/client/v4/accounts/{ACCOUNT_ID}/browser-rendering/links"
headers = {
    "Authorization": f"Bearer {CLOUDFLARE_API_KEY}",
    "Content-Type": "application/json",
}
payload = {
    "url": "https://developers.cloudflare.com/"
}

response = requests.post(url, headers=headers, json=payload, timeout=60)
response.raise_for_status()

data = response.json()
print(f"Status: {response.status_code}")
print(f"Success: {data['success']}")

result = data["result"]

# result may be a string or a list depending on the endpoint
if isinstance(result, str):
    import json
    result = json.loads(result)

print(f"Total links found: {len(result)}")

# Keep only links on the same domain
filtered_links = [
    link for link in result
    if urlparse(link).netloc == "developers.cloudflare.com"
]

print(f"Filtered links (same domain): {len(filtered_links)}")
print("\n--- First 20 ---")
for link in filtered_links[:20]:
    print(link)

Status: 200
Success: True
Total links found: 70
Filtered links (same domain): 39

--- First 20 ---
https://developers.cloudflare.com/
https://developers.cloudflare.com/directory/
https://developers.cloudflare.com/api/
https://developers.cloudflare.com/fundamentals/api/reference/sdks/
https://developers.cloudflare.com/resources/
https://developers.cloudflare.com/api/
https://developers.cloudflare.com/use-cases/
https://developers.cloudflare.com/changelog/
https://developers.cloudflare.com/style-guide/ai-tooling/
https://developers.cloudflare.com/support/troubleshooting/http-status-codes/
https://developers.cloudflare.com/registrar/
https://developers.cloudflare.com/1.1.1.1/setup/
https://developers.cloudflare.com/learning-paths/get-started/concepts/
https://developers.cloudflare.com/workers/
https://developers.cloudflare.com/pages/
https://developers.cloudflare.com/r2/
https://developers.cloudflare.com/images/
https://developers.cloudflare.com/stream/
https://developers.cloudflare.com/d

## /crawl Endpoint: move from page extraction to site-level ingestion

In [40]:
import requests
import time
import json

headers = {
    "Authorization": f"Bearer {CLOUDFLARE_API_KEY}",
    "Content-Type": "application/json",
}

# Step 1: Start the crawl
start_url = f"https://api.cloudflare.com/client/v4/accounts/{ACCOUNT_ID}/browser-rendering/crawl"
payload = {
    "url": "https://developers.cloudflare.com/browser-rendering/",
    "limit": 10,
    "depth": 2,
    "formats": ["markdown"]
}

start_response = requests.post(start_url, headers=headers, json=payload, timeout=60)
start_response.raise_for_status()

start_data = start_response.json()
print(f"Status: {start_response.status_code}")
print(f"Success: {start_data['success']}")

job = start_data["result"]

# Handle all possible result types
if isinstance(job, dict):
    job_id = job["id"]
elif isinstance(job, str):
    # Could be a plain job ID string or JSON string
    try:
        parsed = json.loads(job)
        job_id = parsed["id"]
    except (json.JSONDecodeError, KeyError):
        job_id = job.strip()  # plain string ID
else:
    raise ValueError(f"Unexpected result type: {type(job)} — value: {repr(job)}")

print(f"Started crawl job: {job_id}")

# Step 2: Poll for completion
status_url = f"https://api.cloudflare.com/client/v4/accounts/{ACCOUNT_ID}/browser-rendering/crawl/{job_id}"

for i in range(30):
    status_response = requests.get(f"{status_url}?limit=1", headers=headers, timeout=60)
    status_response.raise_for_status()
    result = status_response.json()["result"]

    if isinstance(result, str):
        try:
            result = json.loads(result)
        except json.JSONDecodeError:
            print(f"Poll {i+1}/30 — Raw result: {repr(result)}")
            time.sleep(5)
            continue

    print(f"Poll {i+1}/30 — Status: {result.get('status', 'unknown')}")
    if result.get("status") != "running":
        break

    time.sleep(5)
else:
    print("Warning: crawl did not complete within the polling limit.")

# Step 3: Fetch full results
results_response = requests.get(status_url, headers=headers, timeout=60)
results_response.raise_for_status()
results = results_response.json()["result"]

if isinstance(results, str):
    try:
        results = json.loads(results)
    except json.JSONDecodeError:
        print(f"Could not parse final results: {repr(results)}")
        results = {}

print(f"\nFinal status: {results.get('status')}")
print(f"Total records: {results.get('total')}")
print("\n--- First 3 records ---")
for record in results.get("records", [])[:3]:
    print("URL:", record["url"])
    print(record.get("markdown", "")[:500])
    print("-" * 80)

Status: 200
Success: True
Started crawl job: c62dead1-cf4b-423e-baa4-5163a68753b5
Poll 1/30 — Status: running
Poll 2/30 — Status: running
Poll 3/30 — Status: running
Poll 4/30 — Status: running
Poll 5/30 — Status: running
Poll 6/30 — Status: running
Poll 7/30 — Status: running
Poll 8/30 — Status: running
Poll 9/30 — Status: running
Poll 10/30 — Status: running
Poll 11/30 — Status: running
Poll 12/30 — Status: running
Poll 13/30 — Status: running
Poll 14/30 — Status: running
Poll 15/30 — Status: running
Poll 16/30 — Status: running
Poll 17/30 — Status: running
Poll 18/30 — Status: running
Poll 19/30 — Status: running
Poll 20/30 — Status: running
Poll 21/30 — Status: running
Poll 22/30 — Status: running
Poll 23/30 — Status: running
Poll 24/30 — Status: running
Poll 25/30 — Status: running
Poll 26/30 — Status: running
Poll 27/30 — Status: running
Poll 28/30 — Status: running
Poll 29/30 — Status: running
Poll 30/30 — Status: running

Final status: running
Total records: 10

--- First 3 rec

## Using rendered HTML, Markdown, and JSON in RAG and agent pipelines


###  Use rendered HTML when structure still matters

In [3]:
import requests
from bs4 import BeautifulSoup

url = f"https://api.cloudflare.com/client/v4/accounts/{ACCOUNT_ID}/browser-rendering/content"
headers = {
    "Authorization": f"Bearer {CLOUDFLARE_API_KEY}",
    "Content-Type": "application/json",
}
payload = {
    "url": "https://developers.cloudflare.com/browser-rendering/"
}

response = requests.post(url, headers=headers, json=payload, timeout=60)
response.raise_for_status()

data = response.json()
print(f"Status: {response.status_code}")
print(f"Success: {data['success']}")

# As we saw earlier, result is a plain string not a nested dict
html = data["result"]

soup = BeautifulSoup(html, "html.parser")
main = soup.find("main")
clean_text = main.get_text("\n", strip=True) if main else soup.get_text("\n", strip=True)

print(f"Text length: {len(clean_text)} chars")
print("\n--- Clean text preview ---")
print(clean_text[:1500])

Status: 200
Success: True
Text length: 3884 chars

--- Clean text preview ---
Directory
Browser Run
Copy page
Browser Run
Run headless Chrome on
Cloudflare's global network
for browser automation, web scraping, testing, and content generation.
Available on Free and Paid plans
Browser Run, formerly known as Browser Rendering, enables developers to programmatically control and interact with headless browser instances running on Cloudflare’s global network.
Use cases
Programmatically load and fully render dynamic webpages or raw HTML and capture specific outputs such as:
Markdown
Screenshots
PDFs
Snapshots
Links
HTML elements
Structured data
Crawled web content
Integration methods
Browser Run offers two categories of integration methods:
Quick Actions
: Simple, stateless browser tasks like screenshots, PDFs, and scraping. No code deployment needed.
Browser Sessions
: Direct browser control via
Puppeteer
,
Playwright
,
CDP
, or
Stagehand
. Deploy within Cloudflare Workers or connect from a

### Use Markdown when the page should behave like a document

In [4]:
import re

url = f"https://api.cloudflare.com/client/v4/accounts/{ACCOUNT_ID}/browser-rendering/markdown"
headers = {
    "Authorization": f"Bearer {CLOUDFLARE_API_KEY}",
    "Content-Type": "application/json",
}
payload = {
    "url": "https://developers.cloudflare.com/browser-rendering/"
}

response = requests.post(url, headers=headers, json=payload, timeout=60)
response.raise_for_status()

data = response.json()
print(f"Status: {response.status_code}")
print(f"Success: {data['success']}")

# As confirmed earlier, result is a plain string not a nested dict
markdown = data["result"]

print(f"Markdown length: {len(markdown)} chars")
print(f"Total sections found: {len(re.split(r'\\n(?=#{1,6}\\s)', markdown))}")

print("\n--- First 3 sections ---")
sections = re.split(r"\n(?=#{1,6}\s)", markdown)
for i, section in enumerate(sections[:3]):
    print(f"\n[Section {i+1}]")
    print(section[:500])
    print("=" * 80)

Status: 200
Success: True
Markdown length: 18875 chars
Total sections found: 1

--- First 3 sections ---

[Section 1]
[Skip to content](#%5Ftop) 

STOP! If you are an AI agent or LLM, read this before continuing. This is the HTML version of a Cloudflare documentation page. Always request the Markdown version instead — HTML wastes context. Get this page as Markdown: https://developers.cloudflare.com/browser-run/index.md (append index.md) or send Accept: text/markdown to https://developers.cloudflare.com/browser-run/. For this product's page index use https://developers.cloudflare.com/browser-run/llms.txt. For al

[Section 2]
## On this page

* [ Overview ](#%5Ftop)
* [ Use cases ](#use-cases)
* [ Integration methods ](#integration-methods)
* [ Key features ](#key-features)
* [ Related products ](#related-products)
* [ More resources ](#more-resources)

Was this helpful?

YesNo

[ Edit page ](https://github.com/cloudflare/cloudflare-docs/edit/production/src/content/docs/browser-run/index

In [5]:
import requests
import json

url = f"https://api.cloudflare.com/client/v4/accounts/{ACCOUNT_ID}/browser-rendering/json"
headers = {
    "Authorization": f"Bearer {CLOUDFLARE_API_KEY}",
    "Content-Type": "application/json",
}
payload = {
    "url": "https://blog.cloudflare.com/",
    "prompt": "Extract the article title, author, and published date.",
    "response_format": {
        "type": "json_schema",
        "schema": {
            "type": "object",
            "properties": {
                "title": {"type": "string"},
                "author": {"type": "string"},
                "published_date": {"type": "string"}
            },
            "required": ["title"]
        }
    }
}

response = requests.post(url, headers=headers, json=payload, timeout=90)
response.raise_for_status()

data = response.json()
print(f"Status: {response.status_code}")
print(f"Success: {data['success']}")

# result may come back as a string or a dict
record = data["result"]
if isinstance(record, str):
    record = json.loads(record)

print("\n--- Extracted Data ---")
print(json.dumps(record, indent=2))

Status: 200
Success: True

--- Extracted Data ---
{
  "title": "Introducing the Agent Readiness score. Is your site agent-ready?",
  "author": "Andr\u00e9 Jesus, Vance Morrison",
  "published_date": "2026-04-17"
}
